# PR to PO Master Notebook

**Purpose:** End-to-end pipeline for PR validation: document upload → **document categorization** (LLM) → **document parsing** (LLM + JSON schema examples, with retries and validation) → **database** persistence → **Compliance Check 1** (attachment existence & classification) → **Compliance Check 2** (document validity & applicability).

**Environment:** The notebook detects which tech stack is available: **Production (AWS)** — Bedrock, RDS, Titan embeddings — or **Local** — Groq LLM (free tier), local embeddings, SQLite (default; use [DBeaver](https://dbeaver.io/) to view) or optional PostgreSQL. Run all cells in order; later sections depend on variables set in earlier ones (e.g. `schemas`, `uploaded`, `classified_documents`, `parsed_quotes`).

**Getting started (steps to run the notebook)**

1. **Folders:** The notebook uses **input** (documents in) and **output** (extracted JSONs out). They are created under `document_processing_rag/` when you run Section 1. Put your PDFs, Word, or Excel files in **input** before running Section 3.
2. **Environment:** Set **GROQ_API_KEY** (for local LLM) or AWS vars for production. **Database:** If you omit **DATABASE_URL**, the notebook uses SQLite (`pr_validation.db` in BASE_DIR). Use [DBeaver](https://dbeaver.io/) to open and browse the SQLite file. Optional: set **DATABASE_URL** for PostgreSQL and connect with DBeaver to localhost.
3. **Run:** Open the notebook, run Section 0 (env detection) → Section 1 (paths, LLM) → Section 2 (schemas) → Section 3 (list from input) → … through Section 10 (write outputs to output folder). All extracted JSONs and check results are in **output**; DB (SQLite by default, or PostgreSQL) holds the same data — open with DBeaver to inspect.

**Notebook verification (what’s implemented):**  
- **§0–§1:** Env detection and imports; LLM is Groq (local) or Bedrock (production).  
- **§2:** All extraction schemas and Check 1/2 result schemas loaded; each schema’s **example** is used in prompts.  
- **§3–§4:** Document list from synthetic folder; each file classified with **CLASSIFY_SYSTEM** + JSON example.  
- **§5:** **Quote** and **MSA** parsing use **system prompts** that embed the full schema **example**; **run_extraction_with_retries** (retries, **validate_parsed_output**, **merge_deep**); **parse_quote_with_llm** / **parse_msa_with_llm** are one-line wrappers.  
- **Optional:** LangGraph extract → validate → retry-or-end using the same prompts and validation.  
- **§6–§9:** DB persistence; Check 1 (attachment policy); Check 2 (quote validity); summary print.

---
## 0. Environment detection

**What this does:** Reads environment variables to choose the tech stack so the same notebook runs locally (Groq API key) or in production (AWS + Bedrock model ID). **Production** requires `AWS_REGION` or `AWS_PROFILE` and `BEDROCK_MODEL_ID`; **Local** uses `GROQ_API_KEY` and optionally `DATABASE_URL` or `LOCAL_DB_PATH` (default SQLite). Sets `ENV_MODE`, `USE_BEDROCK`, `USE_GROQ`, and `DATABASE_URL` for later cells.

In [ ]:
import os
import json

def detect_env():
    use_aws = (
        os.environ.get("AWS_REGION") or os.environ.get("AWS_PROFILE")
    ) and os.environ.get("BEDROCK_MODEL_ID")
    use_groq = bool(os.environ.get("GROQ_API_KEY"))
    
    if use_aws:
        return "aws"
    if use_groq:
        return "local"
    return "local"  # default to local for notebook

ENV_MODE = detect_env()
print(f"Detected mode: {ENV_MODE}")

# LLM: Bedrock (AWS) or Groq (local)
USE_BEDROCK = ENV_MODE == "aws"
USE_GROQ = ENV_MODE == "local"

# Embeddings: Titan (AWS) or sentence-transformers (local)
USE_TITAN = ENV_MODE == "aws"
USE_LOCAL_EMBEDDINGS = ENV_MODE == "local"

# DB: RDS (AWS) or SQLite/Postgres (local)
DATABASE_URL = os.environ.get("DATABASE_URL") or os.environ.get("LOCAL_DB_PATH") or "sqlite:///pr_validation.db"
print(f"Database: {DATABASE_URL}")

---
## 1. Imports and config

**What this does:** Loads `.env` (if present), resolves **BASE_DIR** / **SCHEMAS_DIR** / **INPUT_DIR** / **OUTPUT_DIR** so the notebook finds `schemas/`, **input** (documents in), and **output** (extracted JSONs out). Creates **input** and **output** folders if missing. Then creates the **LLM** (`ChatGroq` or `ChatBedrock`) used for classification and parsing. Run this cell after Section 0.

In [ ]:
# Optional: load .env if present
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

import os
from pathlib import Path

# Base paths: run from repo root (PR to PO Agent) or from document_processing_rag
CWD = Path(os.getcwd())
if (CWD / "schemas").exists():
    BASE_DIR = CWD
elif (CWD / "document_processing_rag" / "schemas").exists():
    BASE_DIR = CWD / "document_processing_rag"
else:
    BASE_DIR = CWD

SCHEMAS_DIR = BASE_DIR / "schemas"
# Input: place PDFs/Word/Excel here for processing
INPUT_DIR = BASE_DIR / "input"
# Output: extracted JSONs and check results written here
OUTPUT_DIR = BASE_DIR / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Fallback if input is empty: synthetic_data/multi item example/live
if (BASE_DIR.parent / "synthetic_data").exists():
    SYNTHETIC_DIR = BASE_DIR.parent / "synthetic_data" / "multi item example" / "live"
else:
    SYNTHETIC_DIR = BASE_DIR / "synthetic_data" / "multi item example" / "live"

print("BASE_DIR:", BASE_DIR)
print("SCHEMAS_DIR:", SCHEMAS_DIR, "exists:", SCHEMAS_DIR.exists())
print("INPUT_DIR:", INPUT_DIR, "exists:", INPUT_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR, "exists:", OUTPUT_DIR.exists())

In [ ]:
# LangChain and LLM
try:
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_core.output_parsers import JsonOutputParser
except ImportError:
    raise ImportError("Install: pip install langchain-core")

if USE_GROQ:
    try:
        from langchain_groq import ChatGroq
        llm = ChatGroq(
            model=os.environ.get("LOCAL_LLM_MODEL", "llama-3-8b-8192"),
            api_key=os.environ.get("GROQ_API_KEY"),
            temperature=0.1,
        )
        print("LLM: Groq (local)")
    except ImportError:
        raise ImportError("Install: pip install langchain-groq")
elif USE_BEDROCK:
    try:
        from langchain_aws import ChatBedrock
        llm = ChatBedrock(
            model_id=os.environ.get("BEDROCK_MODEL_ID", "anthropic.claude-3-5-haiku-20241022-v2:0"),
            region_name=os.environ.get("AWS_REGION", "us-east-1"),
            temperature=0.1,
        )
        print("LLM: Bedrock (AWS)")
    except ImportError:
        raise ImportError("Install: pip install langchain-aws boto3")
else:
    raise RuntimeError("Set GROQ_API_KEY (local) or AWS + BEDROCK_MODEL_ID (production)")

---
## 2. Load schema references

**What this does:** Loads all extraction and check-result schemas from `schemas/` so the rest of the notebook can use them as **templates and one-shot examples** for the LLM.

- **`load_schema(name)`** — Loads `schemas/{name}.json` (e.g. `pr_header`, `quote`, `msa`). Each file contains a JSON Schema and an **`example`** object; the extraction system prompts embed that example so the LLM outputs matching structure.
- **`load_check_result(check_num)`** — Loads the result schema for Check 1 or Check 2 from `schemas/compliance_checks/check_0X_.../check_0X_result.json`.

**Output:** A dict `schemas` with keys `pr_header`, `pr_line_items`, `pr_attachments`, `quote`, `msa`, plus `check1_schema` and `check2_schema`. All files live under **SCHEMAS_DIR** (`document_processing_rag/schemas/`). **Tie-in:** (1) **quote.json** / **msa.json** — `example` is embedded in extraction system prompts; `required` and `properties.line_items.items.required` drive **validate_parsed_output**. (2) **pr_header.json**, **pr_line_items.json**, **pr_attachments.json** — `example` is used for stub PR data (Section 5 stub cell); pr_attachments stub maps classifier document_type to schema **document_classification** enum. (3) **compliance_checks/check_01_.../check_01_result.json** and **check_02_.../check_02_result.json** — `description` and `example` build check system prompts; `required` drives **validate_check_output**.

In [ ]:
def load_schema(name: str) -> dict:
    path = SCHEMAS_DIR / f"{name}.json"
    if not path.exists():
        raise FileNotFoundError(str(path))
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_check_result(check_num: int) -> dict:
    if check_num == 1:
        path = SCHEMAS_DIR / "compliance_checks" / "check_01_attachment_existence_classification" / "check_01_result.json"
    elif check_num == 2:
        path = SCHEMAS_DIR / "compliance_checks" / "check_02_document_validity_applicability" / "check_02_result.json"
    else:
        raise ValueError(f"Check {check_num} schema not yet added")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# Load extraction schemas (for parsing)
schemas = {}
for name in ["pr_header", "pr_line_items", "pr_attachments", "quote", "msa"]:
    try:
        schemas[name] = load_schema(name)
        print(f"Loaded schema: {name}")
    except FileNotFoundError as e:
        print(f"Skip {name}: {e}")

# Check result schemas
check1_schema = load_check_result(1)
check2_schema = load_check_result(2)
print("Check 1 & 2 result schemas loaded.")

---
## 3. Document upload

**What this does:** Simulates the “PR upload” step by **listing documents** in a folder (by default `synthetic_data/multi item example/live`). In production this would be replaced by S3 keys or in-memory bytes from an API.

- **`list_uploaded_documents(source_dir)`** — Returns a list of dicts `{path, filename, size_bytes}` for each PDF, Word, or Excel file in the folder. Used so the pipeline knows which files to classify and parse.

**Output:** Variable `uploaded` — a list of document metadata. If the folder is missing or empty, `uploaded` will be empty and later steps will have nothing to process; place sample PDFs (e.g. quote and MSA) in the synthetic folder before running.

In [ ]:
def list_uploaded_documents(source_dir: Path = None) -> list[dict]:
    """List documents available for processing. Each item: {path, filename, size_bytes}."""
    source_dir = source_dir or INPUT_DIR
    if not source_dir.exists():
        source_dir = SYNTHETIC_DIR
    if not source_dir.exists():
        return []
    out = []
    for f in source_dir.iterdir():
        if f.is_file() and f.suffix.lower() in (".pdf", ".docx", ".xlsx", ".doc", ".xls"):
            out.append({
                "path": str(f),
                "filename": f.name,
                "size_bytes": f.stat().st_size,
            })
    return sorted(out, key=lambda x: x["filename"])

uploaded = list_uploaded_documents()
for u in uploaded:
    print(u["filename"], "(", u["size_bytes"], "bytes )")
if not uploaded:
    print("No documents found. Place PDF/Word/Excel in the input folder (document_processing_rag/input) or in synthetic_data/.../live.")

---
## 4. Document categorization (Check 1 input)

**What this does:** Classifies **each uploaded attachment** into a single document type (Quotation, Contract, SOW, Service Agreement, Invoice, BidSummary, Justification, Spec, Other) using the LLM. This is the **gatekeeper** for Check 1 and determines which parser to use later (e.g. Quote vs MSA).

- **`extract_text_from_file(filepath, max_chars)`** — Pulls the first few pages or paragraphs of text from a PDF/Word/Excel file so the LLM has content to classify; falls back to the filename if the library is missing or extraction fails.
- **`CLASSIFY_SYSTEM`** — System prompt that defines the document types and includes a **JSON example** of the expected output (`document_type`, `confidence`, `reason`).
- **`classify_document(llm, filepath, filename, excerpt)`** — Sends the system prompt plus the file name and excerpt to the LLM, parses the response as JSON, and returns the classification (or a safe default on parse failure).

**Output:** `classified_documents` — a list of dicts, one per uploaded file, each with `filename`, `document_type`, `confidence`, and `reason`. This list is consumed by Check 1 and by the parsing loop (which routes Quote vs MSA by `document_type`).

In [ ]:
def extract_text_from_file(filepath: str, max_chars: int = 2000) -> str:
    """Extract first max_chars of text for classification hint."""
    path = Path(filepath)
    suffix = path.suffix.lower()
    try:
        if suffix == ".pdf":
            try:
                import pypdf
                reader = pypdf.PdfReader(filepath)
                text = ""
                for i, page in enumerate(reader.pages):
                    if i >= 3:
                        break
                    text += page.extract_text() or ""
                return (text or path.name)[:max_chars]
            except ImportError:
                return path.name
        if suffix in (".docx", ".doc"):
            try:
                import docx
                doc = docx.Document(filepath)
                return "\n".join(p.text for p in doc.paragraphs[:30])[:max_chars]
            except ImportError:
                return path.name
        if suffix in (".xlsx", ".xls"):
            return path.name
    except Exception as e:
        return path.name + " " + str(e)[:200]
    return path.name

# Classification system prompt includes JSON example (from check_01 / schema pattern)
CLASSIFY_EXAMPLE = json.dumps({"document_type": "Quotation", "confidence": 0.94, "reason": "Keywords: Quote, Valid Until; line items with prices; supplier letterhead."})
CLASSIFY_SYSTEM = """You are a procurement document classifier. Given the file name and a short excerpt, classify into exactly one type: Quotation, Contract, SOW, Service Agreement, Invoice, BidSummary, Justification, Spec, Other.
Quotation: Quote, Quotation, Valid Until, line items with prices.
Contract: Agreement, Contract, Master Service Agreement, parties, effective/expiry.
SOW: Statement of Work, Scope of Work, deliverables, milestones.
Invoice: Invoice, Bill To, Payment Due.
Respond with JSON only. Example output: """ + CLASSIFY_EXAMPLE

def classify_document(llm, filepath: str, filename: str, excerpt: str = None) -> dict:
    excerpt = excerpt or extract_text_from_file(filepath)
    prompt = f"File: {filename}\n\nExcerpt:\n{excerpt[:1500]}\n\nClassify. JSON only."
    msg = [SystemMessage(content=CLASSIFY_SYSTEM), HumanMessage(content=prompt)]
    out = llm.invoke(msg)
    text = out.content if hasattr(out, "content") else str(out)
    try:
        # strip markdown code block if present
        if "```" in text:
            text = text.split("```")[1].replace("json", "").strip()
        return json.loads(text)
    except json.JSONDecodeError:
        return {"document_type": "Other", "confidence": 0.5, "reason": "Parse failed"}

# Run classification on uploaded docs
classified_documents = []
for u in uploaded:
    result = classify_document(llm, u["path"], u["filename"])
    result["filename"] = u["filename"]
    classified_documents.append(result)
    print(u["filename"], "->", result.get("document_type"), "(", result.get("confidence"), ")")

**Why LLM parsing was minimal at first:** The first version used a single user prompt + example to get the pipeline running end-to-end. That doesn’t guarantee **full** extraction or **valid** JSON: the model can omit fields, hallucinate, or return invalid JSON. So we now:

1. **Use a system prompt** so the LLM has a fixed role and rules.
2. **Drive extraction from the JSON template** by telling the LLM exactly which fields to extract (from the schema).
3. **Validate** the parsed output against the schema and **retry once** if required fields are missing.
4. **Retry** (up to 3 attempts) with the list of missing/invalid fields and **merge** the new extraction into the previous one.

**Production approach here:** System prompts embed the full schema **example**; we use **validate_parsed_output** and **merge_deep** in **run_extraction_with_retries**. Optionally the same flow runs as a **LangGraph** (extract → validate → retry or end).

**Helpers used by Section 5 (run the cell below first).**  
- **get_schema_field_list(schema)** — Collects property names from the schema for prompt hints.  
- **_schema_for_validation(schema)** — Strips non–JSON Schema keys (e.g. `example`) so `jsonschema.validate()` can run.  
- **validate_parsed_output(data, schema)** — Checks required fields and optional `jsonschema`; returns `(valid: bool, errors: list)`. Used inside the retry loop and by the LangGraph validate node.  
- **merge_deep(base, update)** — Deep-merges `update` into `base` (fills missing or empty values). Used to combine the initial parse with the retry parse.

In [ ]:
# Schema-driven extraction: field list + validation (production tool)
def get_schema_field_list(schema: dict, prefix: str = "") -> list[str]:
    """Recursively collect property names from a JSON schema for prompt guidance."""
    props = schema.get("properties", {})
    out = []
    for key, val in props.items():
        if isinstance(val, dict) and "properties" in val and key not in ("bill_to", "ship_to", "line_items"):
            out.extend(get_schema_field_list(val, prefix + key + "."))
        elif isinstance(val, dict) and "items" in val and "properties" in val.get("items", {}):
            out.append(f"{prefix}{key}[] (each: {list(val['items'].get('properties', {}).keys())})")
        else:
            out.append(prefix + key)
    return out

def _schema_for_validation(schema: dict) -> dict:
    """Return a copy of schema without non-jsonschema keys (e.g. example) for validate()."""
    s = {k: v for k, v in schema.items() if k not in ("example", "title", "description")}
    if "properties" in s:
        s["properties"] = {k: v for k, v in s["properties"].items()}
    return s

# Validation tool: used in retry loop and by LangGraph
def validate_parsed_output(data: dict, schema: dict) -> tuple[bool, list[str]]:
    """Returns (valid, list of missing/invalid field messages). Production tool for extract loop."""
    errors = []
    required = schema.get("required", [])
    for r in required:
        if r not in data or data[r] is None or data[r] == "":
            errors.append(f"Missing required: {r}")
    if "line_items" in schema.get("properties", {}) and isinstance(data.get("line_items"), list):
        item_schema = schema.get("properties", {}).get("line_items", {}).get("items", {})
        for i, item in enumerate(data["line_items"]):
            for req in item_schema.get("required", []):
                if req not in item or item[req] is None:
                    errors.append(f"line_items[{i}] missing: {req}")
    try:
        import jsonschema
        jsonschema.validate(instance=data, schema=_schema_for_validation(schema))
    except ImportError:
        pass
    except Exception as e:
        if "ValidationError" in type(e).__name__:
            errors.append(str(e))
        elif not errors:
            errors.append(str(e))
    return (len(errors) == 0, errors)

def merge_deep(base: dict, update: dict) -> dict:
    """Deep merge update into base (update fills missing or empty in base). For retry merge."""
    out = dict(base)
    for k, v in update.items():
        if v is None or v == "":
            continue
        if k not in out or out[k] is None or out[k] == "":
            out[k] = v
        elif isinstance(out[k], dict) and isinstance(v, dict):
            out[k] = merge_deep(out[k], v)
        elif isinstance(out[k], list) and isinstance(v, list) and v:
            if not out[k]:
                out[k] = v
            else:
                for i, item in enumerate(v):
                    if i < len(out[k]) and isinstance(out[k][i], dict) and isinstance(item, dict):
                        out[k][i] = merge_deep(out[k][i], item)
                    elif i >= len(out[k]):
                        out[k].append(item)
    return out

---
## 5. Document parsing (JSON templates + retries)

**What this does:** For each attachment classified as **Quotation** or **Contract/MSA**, runs **production extraction** with retries, validation, and merge.

- **System prompts:** `QUOTE_EXTRACT_SYSTEM` and `MSA_EXTRACT_SYSTEM` are built from `schemas/quote.json` and `schemas/msa.json` and **embed the full `example`** (up to ~3800 chars) so the LLM outputs JSON that matches the template.
- **`_parse_json_from_llm(text_out)`** — Strips markdown code fences and parses JSON from the LLM response; returns `None` on failure.
- **`run_extraction_with_retries(llm, doc_type, filepath, filename, max_retries=3)`** — Loop: build prompt (first attempt = document + example snippet; retries = excerpt + validation errors), call LLM, parse JSON, run **`validate_parsed_output(parsed, schema)`**. If invalid and retries left, call LLM again with the error list and **`merge_deep(parsed, retry_parsed)`**; repeat until valid or max retries. Returns the merged parsed dict or a minimal fallback.
- **`parse_quote_with_llm`** / **`parse_msa_with_llm`** — Thin wrappers that call `run_extraction_with_retries` with `doc_type` `"quote"` or `"msa"`.

**Output:** `parsed_quotes` and `parsed_msas` — lists of extracted JSONs, one per classified quote or MSA attachment.

In [ ]:
# System prompts: schemas/quote.json and schemas/msa.json (loaded in §2); example embeds in prompt, required drives validation
_quote_schema = schemas.get("quote", {})
_quote_example = json.dumps(_quote_schema.get("example", {}), indent=2)[:3800]
QUOTE_EXTRACT_SYSTEM = """You are a quote parser for procurement. Extract structured data from supplier quotation text into a JSON object. Your output MUST match this structure and types. Extract every field that appears in the document.

Example output (adapt values to the document; keep structure and types):
""" + _quote_example + """

Rules: Output valid JSON only. No markdown. Dates YYYY-MM-DD. Numbers as numbers. Required: quote_number, quote_date, currency, line_items (each with line_number, quantity, unit_of_measure, unit_price, extended_price)."""

_msa_schema = schemas.get("msa", {})
_msa_example = json.dumps(_msa_schema.get("example", {}), indent=2)[:3800]
MSA_EXTRACT_SYSTEM = """You are an MSA/contract parser for procurement. Extract structured data from Master Service Agreement or contract text. Your output MUST match this structure and types.

Example output (adapt values to the document; keep structure and types):
""" + _msa_example + """

Rules: Output valid JSON only. No markdown. Dates YYYY-MM-DD. Required: agreement_id, effective_date, buyer_name, supplier_name."""

def _parse_json_from_llm(text_out: str) -> dict | None:
    if "```" in text_out:
        text_out = text_out.split("```")[1].replace("json", "").strip()
    try:
        return json.loads(text_out)
    except json.JSONDecodeError:
        return None

def run_extraction_with_retries(llm, doc_type: str, filepath: str, filename: str, max_retries: int = 3) -> dict:
    """Production: retry loop + validation tool + merge. doc_type in ('quote', 'msa')."""
    schema = _quote_schema if doc_type == "quote" else _msa_schema
    system_prompt = QUOTE_EXTRACT_SYSTEM if doc_type == "quote" else MSA_EXTRACT_SYSTEM
    max_chars = 6000 if doc_type == "quote" else 5000
    text = extract_text_from_file(filepath, max_chars=max_chars)
    fallback = (
        {"source_file_name": filename, "quote_number": "unknown", "quote_date": "", "currency": "USD", "line_items": []}
        if doc_type == "quote"
        else {"source_file_name": filename, "agreement_id": "unknown", "effective_date": "", "buyer_name": "", "supplier_name": ""}
    )
    example_snippet = json.dumps(schema.get("example", {}), indent=2)[:2800]
    parsed = None
    errors = []
    for attempt in range(max_retries):
        if attempt == 0:
            prompt = f"Extract data into JSON. Example structure:\n{example_snippet}\n\nDocument (filename: {filename}):\n{text}"
        else:
            prompt = f"Document (excerpt):\n{text[:2500]}\n\nValidation failed (attempt {attempt + 1}/{max_retries}): {errors}\nReturn complete valid JSON that fixes these. Output only JSON."
        msg = [SystemMessage(content=system_prompt), HumanMessage(content=prompt)]
        out = llm.invoke(msg)
        text_out = out.content if hasattr(out, "content") else str(out)
        parsed = _parse_json_from_llm(text_out)
        if not parsed:
            continue
        parsed.setdefault("source_file_name", filename)
        valid, errors = validate_parsed_output(parsed, schema)
        if valid:
            break
        if attempt < max_retries - 1:
            retry_prompt = f"Document (excerpt):\n{text[:2500]}\n\nValidation failed (attempt {attempt + 1}/{max_retries}): {errors}\nReturn complete valid JSON that fixes these. Output only JSON."
            retry_out = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=retry_prompt)])
            retry_parsed = _parse_json_from_llm(retry_out.content if hasattr(retry_out, "content") else str(retry_out))
            if retry_parsed:
                parsed = merge_deep(parsed, retry_parsed)
                valid, errors = validate_parsed_output(parsed, schema)
                if valid:
                    break
    return parsed if parsed else fallback

def parse_quote_with_llm(llm, filepath: str, filename: str) -> dict:
    return run_extraction_with_retries(llm, "quote", filepath, filename, max_retries=3)

def parse_msa_with_llm(llm, filepath: str, filename: str) -> dict:
    return run_extraction_with_retries(llm, "msa", filepath, filename, max_retries=3)

# Parse quotes and MSAs from classified docs (production: retries + validation + merge)
parsed_quotes = []
parsed_msas = []
for u, cl in zip(uploaded, classified_documents):
    dt = cl.get("document_type", "Other")
    if dt == "Quotation":
        parsed_quotes.append(parse_quote_with_llm(llm, u["path"], u["filename"]))
    elif dt in ("Contract", "Master Service Agreement") or "Contract" in dt:
        parsed_msas.append(parse_msa_with_llm(llm, u["path"], u["filename"]))

print("Parsed quotes:", len(parsed_quotes))
print("Parsed MSAs:", len(parsed_msas))

**Optional: LangGraph extraction (production agent with tools and loop)**

**What this does:** Runs the same extraction flow as a **state graph** instead of a simple loop.  
- **ExtractState** holds `doc_type`, `filepath`, `filename`, `text`, `system_prompt`, `schema`, `parsed`, `valid`, `errors`, `attempt`, `max_retries`.  
- **extract_node** — Builds the prompt (first time: full doc; later: excerpt + validation errors), invokes the LLM, parses JSON, sets `parsed` and increments `attempt`.  
- **validate_node** — Calls **validate_parsed_output(parsed, schema)** and returns `valid` and `errors`.  
- **should_retry** — If `valid` → end; else if `attempt < max_retries` → back to **extract**; else end.  
Uses the same **QUOTE_EXTRACT_SYSTEM** / **MSA_EXTRACT_SYSTEM** (with embedded JSON examples) and the same validation tool. The example at the bottom runs the graph on the first classified quote document when LangGraph is installed.

In [ ]:
# LangGraph extraction agent (production: graph with extract → validate → retry or end)
try:
    from langgraph.graph import StateGraph, END
    from typing import TypedDict, Annotated
    LANGGRAPH_AVAILABLE = True
except ImportError:
    LANGGRAPH_AVAILABLE = False

if LANGGRAPH_AVAILABLE:
    class ExtractState(TypedDict):
        doc_type: str
        filepath: str
        filename: str
        text: str
        system_prompt: str
        schema: dict
        parsed: dict | None
        valid: bool
        errors: list
        attempt: int
        max_retries: int

    def extract_node(state: ExtractState) -> dict:
        attempt = state["attempt"]
        prompt = f"Extract data into JSON. Document (filename: {state['filename']}):\n{state['text'][:4000]}" if attempt == 0 else f"Document excerpt:\n{state['text'][:2500]}\n\nValidation failed: {state['errors']}. Return complete valid JSON. Output only JSON."
        msg = [SystemMessage(content=state["system_prompt"]), HumanMessage(content=prompt)]
        out = llm.invoke(msg)
        text_out = out.content if hasattr(out, "content") else str(out)
        parsed = _parse_json_from_llm(text_out)
        if parsed:
            parsed.setdefault("source_file_name", state["filename"])
        return {"parsed": parsed, "attempt": attempt + 1}

    def validate_node(state: ExtractState) -> dict:
        parsed = state.get("parsed")
        if not parsed:
            return {"valid": False, "errors": ["No JSON parsed"]}
        valid, errors = validate_parsed_output(parsed, state["schema"])
        return {"valid": valid, "errors": errors}

    def should_retry(state: ExtractState) -> str:
        if state.get("valid"):
            return "end"
        if state["attempt"] < state["max_retries"]:
            return "extract"
        return "end"

    def build_extraction_graph(doc_type: str):
        schema = _quote_schema if doc_type == "quote" else _msa_schema
        system_prompt = QUOTE_EXTRACT_SYSTEM if doc_type == "quote" else MSA_EXTRACT_SYSTEM
        graph = StateGraph(ExtractState)
        graph.add_node("extract", extract_node)
        graph.add_node("validate", validate_node)
        graph.set_entry_point("extract")
        graph.add_edge("extract", "validate")
        graph.add_conditional_edges("validate", should_retry, {"extract": "extract", "end": END})
        return graph.compile()

    # Example: run graph for first quote doc (if any)
    if uploaded and LANGGRAPH_AVAILABLE:
        q_doc = next((u for u, cl in zip(uploaded, classified_documents) if cl.get("document_type") == "Quotation"), None)
        if q_doc:
            g = build_extraction_graph("quote")
            initial = {"doc_type": "quote", "filepath": q_doc["path"], "filename": q_doc["filename"], "text": extract_text_from_file(q_doc["path"], 6000), "system_prompt": QUOTE_EXTRACT_SYSTEM, "schema": _quote_schema, "parsed": None, "valid": False, "errors": [], "attempt": 0, "max_retries": 3}
            final = g.invoke(initial)
            print("LangGraph extraction (quote) final valid:", final.get("valid"), "attempts:", final.get("attempt"))
else:
    print("LangGraph not installed. pip install langgraph to use graph-based extraction.")

In [ ]:
# Stub PR header and line items (from form or system). Use schema examples from schemas/ so structure matches.
pr_header = schemas["pr_header"].get("example", {})
pr_line_items = schemas["pr_line_items"].get("example", {})
# Map classifier document_type to pr_attachments schema enum (schemas/pr_attachments.json)
DOC_TYPE_TO_ATTACHMENT_CLASS = {
    "Quotation": "Supplier Quotation",
    "Contract": "Master Service Agreement / Contract Services",
    "Master Service Agreement": "Master Service Agreement / Contract Services",
    "Service Agreement": "Master Service Agreement / Contract Services",
    "SOW": "SOW",
    "Invoice": "Invoice",
    "BidSummary": "Competitive Quote Comparison",
    "Justification": "Technical Evaluation / Business Case",
    "Spec": "Other",
}
pr_attachments = {
    "pr_number": pr_header.get("pr_number", "PR-2026-007842"),
    "attachment_count": len(uploaded),
    "attachments": [
        {
            "attachment_number": i + 1,
            "file_name": u["filename"],
            "file_type": "PDF" if u["filename"].endswith(".pdf") else "Excel" if u["filename"].lower().endswith((".xlsx", ".xls")) else "Word" if u["filename"].lower().endswith((".docx", ".doc")) else "Other",
            "document_classification": DOC_TYPE_TO_ATTACHMENT_CLASS.get(cl.get("document_type"), "Other"),
            "classification_confidence": cl.get("confidence", 0.8),
        }
        for i, (u, cl) in enumerate(zip(uploaded, classified_documents))
    ],
}
print("PR header pr_number:", pr_header.get("pr_number"))
print("PR line items count:", len(pr_line_items.get("line_items", [])))
print("PR attachments count:", pr_attachments["attachment_count"])

---
## 6. Database structured entry

**What this does:** Persists pipeline outputs for audit and downstream validation.

- **get_engine()** — Returns a SQLAlchemy engine for `DATABASE_URL` (SQLite or PostgreSQL). For SQLite, the file is created under **BASE_DIR** (e.g. `pr_validation.db`).
- **init_db(engine)** — Creates tables if not present: **pr_header**, **pr_line_items**, **pr_attachments**, **parsed_quote**, **parsed_msa** (each with `id`, JSON/text columns, and `created_at`).
- The cell then inserts the current **pr_header**, **pr_line_items**, **pr_attachments**, and each item in **parsed_quotes** and **parsed_msas** into the DB. PR rows are keyed by **pr_number**; quote/MSA rows store **source_file** and **data_json**.

**Output:** Database is populated (PostgreSQL or SQLite). Extracted JSONs are also written to the **output** folder in Section 10.

In [ ]:
try:
    from sqlalchemy import create_engine, text
    from sqlalchemy.engine import Engine
except ImportError:
    create_engine = None
    print("Install sqlalchemy for DB persistence: pip install sqlalchemy")

USE_POSTGRES = DATABASE_URL.strip().lower().startswith("postgresql")

def get_engine() -> Engine:
    url = DATABASE_URL
    if url.startswith("sqlite") and ("pr_validation" in url or url == "sqlite:///pr_validation.db"):
        url = "sqlite:///" + str(BASE_DIR / "pr_validation.db")
    return create_engine(url, echo=False)

def init_db(engine):
    if USE_POSTGRES:
        with engine.connect() as conn:
            conn.execute(text("""CREATE TABLE IF NOT EXISTS pr_header (id SERIAL PRIMARY KEY, pr_number TEXT UNIQUE, data_json TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)"""))
            conn.execute(text("""CREATE TABLE IF NOT EXISTS pr_line_items (id SERIAL PRIMARY KEY, pr_number TEXT, data_json TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)"""))
            conn.execute(text("""CREATE TABLE IF NOT EXISTS pr_attachments (id SERIAL PRIMARY KEY, pr_number TEXT, data_json TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)"""))
            conn.execute(text("""CREATE TABLE IF NOT EXISTS parsed_quote (id SERIAL PRIMARY KEY, source_file TEXT, data_json TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)"""))
            conn.execute(text("""CREATE TABLE IF NOT EXISTS parsed_msa (id SERIAL PRIMARY KEY, source_file TEXT, data_json TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)"""))
            conn.commit()
    else:
        with engine.connect() as conn:
            conn.execute(text("""CREATE TABLE IF NOT EXISTS pr_header (id INTEGER PRIMARY KEY AUTOINCREMENT, pr_number TEXT UNIQUE, data_json TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)"""))
            conn.execute(text("""CREATE TABLE IF NOT EXISTS pr_line_items (id INTEGER PRIMARY KEY AUTOINCREMENT, pr_number TEXT, data_json TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)"""))
            conn.execute(text("""CREATE TABLE IF NOT EXISTS pr_attachments (id INTEGER PRIMARY KEY AUTOINCREMENT, pr_number TEXT, data_json TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)"""))
            conn.execute(text("""CREATE TABLE IF NOT EXISTS parsed_quote (id INTEGER PRIMARY KEY AUTOINCREMENT, source_file TEXT, data_json TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)"""))
            conn.execute(text("""CREATE TABLE IF NOT EXISTS parsed_msa (id INTEGER PRIMARY KEY AUTOINCREMENT, source_file TEXT, data_json TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)"""))
            conn.commit()

if create_engine:
    engine = get_engine()
    init_db(engine)
    pr_num = pr_header.get("pr_number", "PR-2026-007842")
    with engine.connect() as conn:
        if USE_POSTGRES:
            conn.execute(text("""INSERT INTO pr_header (pr_number, data_json) VALUES (:pn, :j) ON CONFLICT (pr_number) DO UPDATE SET data_json = EXCLUDED.data_json"""), {"pn": pr_num, "j": json.dumps(pr_header)})
        else:
            conn.execute(text("INSERT OR REPLACE INTO pr_header (pr_number, data_json) VALUES (:pn, :j)"), {"pn": pr_num, "j": json.dumps(pr_header)})
        conn.execute(text("INSERT INTO pr_line_items (pr_number, data_json) VALUES (:pn, :j)"), {"pn": pr_num, "j": json.dumps(pr_line_items)})
        conn.execute(text("INSERT INTO pr_attachments (pr_number, data_json) VALUES (:pn, :j)"), {"pn": pr_num, "j": json.dumps(pr_attachments)})
        for q in parsed_quotes:
            conn.execute(text("INSERT INTO parsed_quote (source_file, data_json) VALUES (:f, :j)"), {"f": q.get("source_file_name", ""), "j": json.dumps(q)})
        for m in parsed_msas:
            conn.execute(text("INSERT INTO parsed_msa (source_file, data_json) VALUES (:f, :j)"), {"f": m.get("source_file_name", ""), "j": json.dumps(m)})
        conn.commit()
    print("DB: pr_header, pr_line_items, pr_attachments, parsed_quote, parsed_msa written.", "(PostgreSQL)" if USE_POSTGRES else "(SQLite)")

**Check system prompts and result template (scalable pattern)**

Every compliance check uses: (1) a **system prompt** that describes the check’s purpose and policy rules, and (2) the check result **template** (the `example` from `check_0X_result.json`) so the LLM returns JSON that matches the schema. This lets you add more checks later by adding a new result schema and a new system prompt.

- **build_check_system_prompt(check_schema)** — Builds a system prompt from the schema’s `description` and `example` (output structure).
- **CHECK_1_SYSTEM** / **CHECK_2_SYSTEM** — Built from **check1_schema** and **check2_schema**; used by **run_check_with_llm**.
- **run_check_with_llm(llm, check_num, context_dict)** — Sends system prompt + context to the LLM; parses JSON; validates required keys and status enum; on failure, **one retry** with validation errors in the user message; returns result dict or `None` (then use deterministic fallback).

**Schema locations:** The result **template** is the **example** from `schemas/compliance_checks/check_01_attachment_existence_classification/check_01_result.json` and `check_02_document_validity_applicability/check_02_result.json`. To add Check 3+: add a folder `check_0N_.../check_0N_result.json` with **example**, extend **load_check_result(N)** if needed, then call **run_check_with_llm(llm, N, context)**.

In [ ]:
# Build check system prompts from schema (description + example template). Reuse for any future check.
def build_check_system_prompt(check_schema: dict, policy_rules_text: str = "") -> str:
    """Build system prompt for a compliance check. Uses schema description and example as output template."""
    desc = check_schema.get("description", "Compliance check. Output valid JSON only.")
    example = check_schema.get("example", {})
    example_json = json.dumps(example, indent=2)[:4000]
    prompt = f"""You are a procurement compliance checker. {desc}

Output MUST be valid JSON matching this structure (adapt values to the context provided by the user). No markdown, no code fence.

Example output structure:
{example_json}

Rules: Output only the JSON object. Use exactly the keys shown. Status values: PASS, FAIL, or NEEDS_REVIEW."""
    if policy_rules_text:
        prompt += "\n\nPolicy rules to apply:\n" + policy_rules_text
    return prompt

# Check 1: Attachment Existence & Classification — policy rules and template from check_01_result.json
CHECK_1_POLICY = """- PR must have at least one attachment. Value >= 5000 requires at least 1 quote or contract; value >= 50000 may require 3 quotes or one contract (contract can substitute). Valid document types: Quotation, Contract, SOW, Service Agreement. Invoice on PR is a red flag (after-the-fact). Confidence < 0.8 on any classification → NEEDS_REVIEW."""
CHECK_1_SYSTEM = build_check_system_prompt(check1_schema, CHECK_1_POLICY)

# Check 2: Document Validity & Applicability — template from check_02_result.json
CHECK_2_POLICY = """- Quote date must be <= PR date. Quote valid_through must be >= PR date. Currency and total should match PR (within tolerance). Supplier on quote should match PR supplier. PASS = all pass; FAIL = expired or wrong date; NEEDS_REVIEW = fuzzy match or approaching expiry."""
CHECK_2_SYSTEM = build_check_system_prompt(check2_schema, CHECK_2_POLICY)

def get_check_validation_errors(parsed: dict, check_schema: dict) -> list[str]:
    """Return list of validation errors (missing required, invalid status). Used for retry prompt."""
    errors = []
    if not parsed or not isinstance(parsed, dict):
        return ["No valid JSON object returned"]
    required = check_schema.get("required", [])
    for r in required:
        if r not in parsed or parsed[r] is None:
            errors.append(f"Missing required: {r}")
    status_key = "check_1_status" if "check_1_status" in required else "check_2_status"
    if status_key in required and status_key in parsed:
        if parsed[status_key] not in ("PASS", "FAIL", "NEEDS_REVIEW"):
            errors.append(f"Invalid {status_key}: must be PASS, FAIL, or NEEDS_REVIEW")
    return errors

def validate_check_output(parsed: dict, check_schema: dict) -> bool:
    """Ensure parsed result has required keys for this check. Used after LLM response."""
    return len(get_check_validation_errors(parsed, check_schema)) == 0

def run_check_with_llm(llm, check_num: int, context_dict: dict, max_context_chars: int = 8000) -> dict | None:
    """Run a compliance check using LLM + system prompt + result template. One retry with validation errors. Returns result dict or None (use deterministic fallback)."""
    if check_num == 1:
        schema = check1_schema
        system_prompt = CHECK_1_SYSTEM
    elif check_num == 2:
        schema = check2_schema
        system_prompt = CHECK_2_SYSTEM
    else:
        schema = load_check_result(check_num)
        system_prompt = build_check_system_prompt(schema)
    context_str = json.dumps(context_dict, indent=2)
    if len(context_str) > max_context_chars:
        context_str = context_str[:max_context_chars] + "\n... (truncated)"
    user_msg = f"Context for this compliance check:\n{context_str}\n\nProduce the check result JSON matching the required structure. Output only valid JSON."
    msg = [SystemMessage(content=system_prompt), HumanMessage(content=user_msg)]
    out = llm.invoke(msg)
    text_out = out.content if hasattr(out, "content") else str(out)
    parsed = _parse_json_from_llm(text_out)
    if parsed and validate_check_output(parsed, schema):
        return parsed
    # One retry with validation errors
    errors = get_check_validation_errors(parsed, schema) if parsed else ["Failed to parse JSON"]
    retry_msg = f"Context:\n{context_str}\n\nPrevious attempt had errors: {errors}\nReturn complete valid JSON that fixes these. Output only JSON."
    retry_out = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=retry_msg)])
    retry_text = retry_out.content if hasattr(retry_out, "content") else str(retry_out)
    retry_parsed = _parse_json_from_llm(retry_text)
    if retry_parsed and validate_check_output(retry_parsed, schema):
        return retry_parsed
    return None

---
## 7. PR Validation — Check 1: Attachment Existence & Classification

**What this does:** Evaluates whether the PR has the **right attachments** for its value band.

- **Thresholds:** `THRESHOLD_QUOTE_ABOVE` (e.g. 5000) → at least one quote or contract required; `THRESHOLD_COMPETITIVE` (e.g. 50000) → 3 quotes or one contract. **CONTRACT_SATISFIES_QUOTES** allows a contract to substitute for multiple quotes.
- **Primary:** **run_check_with_llm(llm, 1, context_check1)** uses **CHECK_1_SYSTEM** (system prompt with policy rules + result template from **check_01_result.json**). Context = pr_header, classified_documents, policy_rules. LLM returns full check result JSON; validated with **validate_check_output**.
- **Fallback:** **run_check_1(pr_header, classified_documents)** — deterministic logic if LLM parse/validation fails.
- **Status:** PASS / FAIL / NEEDS_REVIEW per policy.

**Output:** **check1_result** — structure matches **check_01_result.json**. Same pattern scales to more checks (add schema + system prompt, call **run_check_with_llm**).

In [ ]:
# Threshold policy (from config / RDS in production)
THRESHOLD_QUOTE_ABOVE = 5000      # value above which at least 1 quote required
THRESHOLD_COMPETITIVE = 50000    # value above which 3 quotes or contract
CONTRACT_SATISFIES_QUOTES = True # valid contract can substitute for 3 quotes

def run_check_1(pr_header: dict, classified_documents: list) -> dict:
    total = float(pr_header.get("total_estimated_value", 0))
    n_attachments = len(classified_documents)
    valid_types = {"Quotation", "Contract", "SOW", "Service Agreement"}
    valid_support = sum(1 for c in classified_documents if c.get("document_type") in valid_types)
    invoice_detected = any(c.get("document_type") == "Invoice" for c in classified_documents)
    missing = []
    if total >= THRESHOLD_QUOTE_ABOVE and valid_support < 1:
        missing.append("Requires at least 1 quote or contract, none found")
    if total >= THRESHOLD_COMPETITIVE and valid_support < 2 and not CONTRACT_SATISFIES_QUOTES:
        missing.append("Requires 3 quotes or active contract for value > 50K")
    policy_met = len(missing) == 0 and n_attachments >= 1
    low_confidence = any(c.get("confidence", 1) < 0.8 for c in classified_documents)
    status = "FAIL" if not policy_met else ("NEEDS_REVIEW" if low_confidence or invoice_detected else "PASS")
    return {
        "check_id": "check_01",
        "check_name": "Attachment Existence & Classification",
        "check_1_status": status,
        "attachments_found": n_attachments,
        "classified_documents": classified_documents,
        "valid_support_docs_found": valid_support,
        "policy_requirements_met": policy_met,
        "missing_requirements": missing,
        "invoice_detected": invoice_detected,
        "sub_checks": [
            {"name": "At least one attachment exists on PR", "passed": n_attachments >= 1},
            {"name": "Agent classified attachment as Quote/SOW/Contract", "passed": valid_support >= 1},
            {"name": "Required documents for value tier present", "passed": policy_met},
        ],
        "field_level_assessment": [
            {"field_name": "Attachment Count", "status": "pass" if n_attachments >= 1 else "fail", "confidence": 1.0, "detail": f"{n_attachments} documents attached"},
            {"field_name": "Quote Document Type", "status": "pass" if valid_support else "fail", "confidence": classified_documents[0].get("confidence", 0.9) if classified_documents else 0, "detail": f"Classified: {[c.get('document_type') for c in classified_documents]}"},
        ],
        "evidence": [
            {"field": "Attachments Found", "extracted_value": f"{n_attachments} documents", "pr_value": "Required: ≥1", "match": "Match" if n_attachments >= 1 else "Mismatch"},
            {"field": "Document Types", "extracted_value": ", ".join(f"{c.get('document_type')} ({c.get('confidence', 0):.0%})" for c in classified_documents), "pr_value": "Expected: Quote or Contract", "match": "Match" if valid_support else "Mismatch"},
        ],
        "policy_reference": "Procurement Policy v3.2, Section 3.1",
    }

# Run Check 1 via LLM (system prompt + template from check_01_result.json); fallback to deterministic
context_check1 = {
    "pr_header": pr_header,
    "classified_documents": classified_documents,
    "policy_rules": {"THRESHOLD_QUOTE_ABOVE": THRESHOLD_QUOTE_ABOVE, "THRESHOLD_COMPETITIVE": THRESHOLD_COMPETITIVE, "CONTRACT_SATISFIES_QUOTES": CONTRACT_SATISFIES_QUOTES},
}
check1_result = run_check_with_llm(llm, 1, context_check1)
if check1_result is None:
    check1_result = run_check_1(pr_header, classified_documents)
    print("Check 1: used deterministic fallback (LLM parse/validation failed).")
else:
    print("Check 1: result from LLM (system prompt + template).")
print("Check 1 status:", check1_result["check_1_status"])
print("Policy requirements met:", check1_result["policy_requirements_met"])
print("Missing:", check1_result.get("missing_requirements", []))

---
## 8. PR Validation — Check 2: Document Validity & Applicability

**What this does:** Checks that the **parsed quote** (and context) is valid and applicable to the PR.

- **Primary:** **run_check_with_llm(llm, 2, context_check2)** uses **CHECK_2_SYSTEM** (system prompt + result template from **check_02_result.json**). Context = pr_header, pr_line_items, parsed_quotes, parsed_msas. LLM returns full check result JSON.
- **Fallback:** **run_check_2(...)** — deterministic logic when no parsed quote or LLM fails. **parse_date(s)** used in fallback for date comparison.
- **Status:** PASS / FAIL / NEEDS_REVIEW. If no **parsed_quotes**, use deterministic only.

**Output:** **check2_result** — structure matches **check_02_result.json**. To add Check 3+ later: add **check_0N_result.json** (with `example`) and call **run_check_with_llm(llm, N, context)**.

In [ ]:
from datetime import datetime

def parse_date(s) -> datetime | None:
    if not s:
        return None
    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%m/%d/%Y", "%d-%b-%Y", "%b %d, %Y"):
        try:
            return datetime.strptime(str(s).strip()[:10], fmt)
        except ValueError:
            continue
    return None

def run_check_2(pr_header: dict, pr_line_items: dict, parsed_quotes: list, parsed_msas: list) -> dict:
    pr_date_s = pr_header.get("pr_created_date", "")
    pr_date = parse_date(pr_date_s)
    pr_total = float(pr_header.get("total_estimated_value", 0))
    pr_currency = pr_header.get("currency", "USD")
    pr_supplier = (pr_line_items.get("line_items") or [{}])[0].get("suggested_supplier_name") or pr_header.get("contract_reference", "")
    pr_company = pr_header.get("company_code", "") or pr_header.get("company_name", "")

    quote_date_ok = quote_expiry_ok = addressee_ok = currency_ok = total_ok = supplier_ok = True
    evidence_list = []

    if not parsed_quotes:
        return {
            "check_id": "check_02",
            "check_name": "Document Validity & Applicability",
            "check_2_status": "NEEDS_REVIEW",
            "summary_description": "No quote parsed; cannot validate validity.",
            "sub_checks": [],
            "field_level_assessment": [],
            "evidence": [],
            "policy_reference": "Procurement Policy v3.2, Section 3.2",
        }

    q = parsed_quotes[0]
    quote_date = parse_date(q.get("quote_date"))
    valid_through = parse_date(q.get("valid_through"))
    quote_total = float(q.get("total") or q.get("subtotal") or 0)
    quote_currency = q.get("currency", "USD")
    quote_supplier = q.get("supplier_name", "")
    quote_customer = (q.get("bill_to") or {}).get("company") or (q.get("ship_to") or {}).get("company") or ""

    if pr_date and quote_date:
        quote_date_ok = quote_date <= pr_date
        evidence_list.append({"field": "Quote Date", "extracted_value": q.get("quote_date", ""), "pr_value": f"PR Date: {pr_date_s}", "match": "Match" if quote_date_ok else "Mismatch"})
    if pr_date and valid_through:
        quote_expiry_ok = valid_through >= pr_date
        days_left = (valid_through - pr_date).days if pr_date and valid_through else 0
        evidence_list.append({"field": "Quote Expiration", "extracted_value": q.get("valid_through", ""), "pr_value": f"Required: After {pr_date_s}", "match": "Match" if quote_expiry_ok else "Partial" if days_left > 0 else "Mismatch"})
    currency_ok = (quote_currency or pr_currency) == (pr_currency or quote_currency)
    total_ok = abs((quote_total or 0) - pr_total) <= max(pr_total * 0.01, 100)
    supplier_ok = bool(quote_supplier and pr_supplier and (quote_supplier.lower() in pr_supplier.lower() or pr_supplier.lower() in quote_supplier.lower()))

    evidence_list.append({"field": "Supplier", "extracted_value": quote_supplier, "pr_value": pr_supplier or "N/A", "match": "Match" if supplier_ok else "Partial"})
    all_pass = quote_date_ok and quote_expiry_ok and currency_ok and total_ok and supplier_ok
    status = "PASS" if all_pass else ("FAIL" if not quote_date_ok or not quote_expiry_ok else "NEEDS_REVIEW")
    summary = "Quote valid and applicable" if all_pass else "Quote date valid but expiration approaching" if quote_expiry_ok and not total_ok else "Quote validity or applicability issue"

    return {
        "check_id": "check_02",
        "check_name": "Document Validity & Applicability",
        "check_2_status": status,
        "summary_description": summary,
        "quotation_validity": {
            "quote_date_sequence_pass": quote_date_ok,
            "quote_not_expired_pass": quote_expiry_ok,
            "quote_staleness_warning": False,
            "quote_addressee_match_pass": addressee_ok,
            "quote_currency_match_pass": currency_ok,
            "quote_total_match_pass": total_ok,
            "quote_supplier_match_pass": supplier_ok,
        },
        "sub_checks": [
            {"name": "Quote date before PR date, expiration after PR date", "passed": quote_date_ok and quote_expiry_ok},
            {"name": "Document applies to same supplier and legal entity", "passed": supplier_ok},
        ],
        "field_level_assessment": [
            {"field_name": "Quote Date", "status": "pass" if quote_date_ok else "fail", "confidence": 0.98, "detail": f"Quote dated {q.get('quote_date')} (before PR date)" if quote_date_ok else "Quote date after PR date"},
            {"field_name": "Quote Expiration Date", "status": "pass" if quote_expiry_ok else "review", "confidence": 0.96, "detail": f"Valid until {q.get('valid_through')}"},
        ],
        "evidence": evidence_list,
        "policy_reference": "Procurement Policy v3.2, Section 3.2",
        "agent_recommendation": None if all_pass else "Verify quote expiration and supplier match." if status == "NEEDS_REVIEW" else "Quote expired or invalid; request new quote.",
    }

# Run Check 2 via LLM (system prompt + template from check_02_result.json); fallback to deterministic
context_check2 = {"pr_header": pr_header, "pr_line_items": pr_line_items, "parsed_quotes": parsed_quotes, "parsed_msas": parsed_msas}
check2_result = run_check_with_llm(llm, 2, context_check2) if parsed_quotes else None
if check2_result is None:
    check2_result = run_check_2(pr_header, pr_line_items, parsed_quotes, parsed_msas)
    print("Check 2: used deterministic fallback (no quote or LLM parse/validation failed).")
else:
    print("Check 2: result from LLM (system prompt + template).")
print("Check 2 status:", check2_result["check_2_status"])
print("Summary:", check2_result.get("summary_description", ""))
qv = check2_result.get("quotation_validity") or {}
print("Quote date OK:", qv.get("quote_date_sequence_pass"))
print("Quote not expired OK:", qv.get("quote_not_expired_pass"))
print("Supplier match OK:", qv.get("quote_supplier_match_pass"))

---
## 9. Summary

**What this does:** Prints a short summary of **check1_result** and **check2_result** (status, description, key flags). In production, these dicts would be sent to a dashboard or validation API.

**Pipeline recap:** Env detection (Section 0) → Imports and paths (Section 1) → Load schemas (Section 2) → List uploaded docs (Section 3) → Classify each doc with LLM (Section 4) → Parse Quote/MSA with schema-driven prompts, retries, validation, and merge (Section 5) → Optional LangGraph extraction → Stub PR header/attachments → DB insert (Section 6) → Check 1 attachment policy (Section 7) → Check 2 quote validity (Section 8). Extend by adding Checks 3–12 and switching to Bedrock/RDS when running in production.

In [ ]:
# Summary output
print("=== Pipeline complete ===")
print("Check 1:", check1_result["check_1_status"], "| policy_requirements_met:", check1_result["policy_requirements_met"])
print("Check 2:", check2_result["check_2_status"], "|", check2_result["summary_description"])
print("\nCheck 1 result keys:", list(check1_result.keys()))
print("Check 2 result keys:", list(check2_result.keys()))

---
## 10. Write outputs to output folder

**What this does:** Writes all extracted and check result JSONs to the **output** folder (`document_processing_rag/output`): **pr_header.json**, **pr_line_items.json**, **pr_attachments.json**, **parsed_quotes.json**, **parsed_msas.json**, **check1_result.json**, **check2_result.json**. Run this cell after Section 9 so the output folder has a full snapshot for downstream use or audit.

In [ ]:
def write_extracted_jsons_to_output(out_dir: Path = None) -> None:
    """Write pr_header, pr_line_items, pr_attachments, parsed_quotes, parsed_msas, check1_result, check2_result to output folder."""
    out_dir = out_dir or OUTPUT_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "pr_header.json", "w", encoding="utf-8") as f:
        json.dump(pr_header, f, indent=2)
    with open(out_dir / "pr_line_items.json", "w", encoding="utf-8") as f:
        json.dump(pr_line_items, f, indent=2)
    with open(out_dir / "pr_attachments.json", "w", encoding="utf-8") as f:
        json.dump(pr_attachments, f, indent=2)
    with open(out_dir / "parsed_quotes.json", "w", encoding="utf-8") as f:
        json.dump(parsed_quotes, f, indent=2)
    with open(out_dir / "parsed_msas.json", "w", encoding="utf-8") as f:
        json.dump(parsed_msas, f, indent=2)
    with open(out_dir / "check1_result.json", "w", encoding="utf-8") as f:
        json.dump(check1_result, f, indent=2)
    with open(out_dir / "check2_result.json", "w", encoding="utf-8") as f:
        json.dump(check2_result, f, indent=2)
    print("Output written to:", out_dir)
    for name in ["pr_header.json", "pr_line_items.json", "pr_attachments.json", "parsed_quotes.json", "parsed_msas.json", "check1_result.json", "check2_result.json"]:
        print("  -", name)
write_extracted_jsons_to_output()